In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install -U gdown

In [ ]:
!pip install py7zr

Loading KArSL

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1R8ZI6a1xgRNqUThDFA-X4RT7FJh9WeRr" -O /content/drive/MyDrive/arabic_dataset/

In [ ]:
import subprocess
import os

BASE   = "/content/drive/MyDrive/arabic_dataset/KArSL-502"
SPLITS = ["train", "test"]

KEEP_FIRST  = [f"{i:04d}" for i in list(range(1, 11)) + list(range(32, 71))]
KEEP_SECOND = [f"{i:04d}" for i in range(71, 81)]

def extract_classes(archive_path, class_list, out_dir, archive_prefix):
    os.makedirs(out_dir, exist_ok=True)
    for cls in class_list:
        print(f"  Extracting class {cls}...")
        result = subprocess.run([
            "7z", "x", archive_path,
            f"{archive_prefix}{cls}\\*",
            f"-o{out_dir}",
            "-y"
        ], capture_output=True, text=True)
        if result.returncode != 0:
            print(f"    WARNING: {result.stderr.strip()}")

for signer in ["01", "02", "03"]:
    for split in SPLITS:
        archive_dir = f"{BASE}/{signer}/{split}"
        out_dir     = f"{BASE}/{signer}/{split}/extracted"

        # check both naming variants
        arch1 = f"{archive_dir}/0001-0070.7z"
        if not os.path.exists(arch1):
            arch1 = f"{archive_dir}/0001-0071.7z"

        arch2 = f"{archive_dir}/0071-0170.7z"

        if os.path.exists(arch1):
            prefix1 = os.path.basename(arch1).replace(".7z", "") + "\\"
            print(f"\n[{signer}/{split}] First archive ({os.path.basename(arch1)})...")
            extract_classes(arch1, KEEP_FIRST, out_dir, prefix1)
        else:
            print(f"\n[{signer}/{split}] ⚠️ First archive not found, skipping...")

        if os.path.exists(arch2):
            print(f"\n[{signer}/{split}] Second archive...")
            extract_classes(arch2, KEEP_SECOND, out_dir, "0071-0170\\")
        else:
            print(f"\n[{signer}/{split}] ⚠️ Second archive not found, skipping...")

    print(f"\n✅ Signer {signer} done!")

print("\n🎉 All signers done!")

In [ ]:
# @title Seeing the structure
import os

base = "/content/drive/MyDrive/arabic_dataset"
for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')

In [ ]:
import os

BASE = "/content/drive/MyDrive/arabic_dataset/KArSL-502"

for signer in ["01", "02", "03"]:
    print(f"\n{'='*40}")
    print(f"  Signer {signer}")
    print(f"{'='*40}")

    for split in ["train", "test"]:
        extracted_path = os.path.join(BASE, signer, split, "extracted")

        if not os.path.exists(extracted_path):
            print(f"  {split}: ❌ extracted folder not found")
            continue

        total_videos = 0
        total_classes = 0

        for cls in sorted(os.listdir(extracted_path)):
            cls_path = os.path.join(extracted_path, cls)
            if not os.path.isdir(cls_path):
                continue
            # each file directly inside the class folder = one video
            videos = len(os.listdir(cls_path))
            total_videos += videos
            total_classes += 1

        print(f"  {split}: {total_classes} classes | {total_videos} videos")

        # show a sample path so we can verify structure
        first_cls = sorted(os.listdir(extracted_path))[0]
        first_video = os.listdir(os.path.join(extracted_path, first_cls))[0]
        print(f"  sample: .../{signer}/{split}/extracted/{first_cls}/{first_video}")

Uploading libraries

In [ ]:
!pip install protobuf==5.29.1 -q
!pip install mediapipe==0.10.21 --no-deps -q

In [ ]:
import google.protobuf
print(google.protobuf.__version__)

import mediapipe as mp
print(mp.__version__)

Defining functions

In [ ]:
import cv2
import numpy as np
import os
import glob
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request

print(f"MediaPipe version: {mp.__version__}")

SEQ_LEN = 30
N_AUG   = 10

# ── Download model files (run once) ──────────────────────────────────────────
def download(url, path):
    if not os.path.exists(path):
        print(f"Downloading {path}...")
        urllib.request.urlretrieve(url, path)

download("https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
         "pose_landmarker.task")
download("https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task",
         "hand_landmarker.task")
download("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task",
         "face_landmarker.task")

# ── Initialize models (Tasks API) ────────────────────────────────────────────
pose_landmarker = vision.PoseLandmarker.create_from_options(
    vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path="pose_landmarker.task"),
        running_mode=vision.RunningMode.IMAGE
    )
)
hand_landmarker = vision.HandLandmarker.create_from_options(
    vision.HandLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path="hand_landmarker.task"),
        running_mode=vision.RunningMode.IMAGE,
        num_hands=2
    )
)
face_landmarker = vision.FaceLandmarker.create_from_options(
    vision.FaceLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path="face_landmarker.task"),
        running_mode=vision.RunningMode.IMAGE,
        num_faces=1
    )
)
print("Models loaded ✓")
# ─────────────────────────────────────────────────────────────────────────────

def read_video(path):
    cap, frames = cv2.VideoCapture(path), []
    while True:
        ret, frame = cap.read()
        if not ret: break
        frames.append(frame)
    cap.release()
    return frames

def augment_video(frames, seed=None):
    rng   = np.random.default_rng(seed)
    angle = rng.uniform(-5, 5)
    scale = rng.uniform(0.8, 1.0)
    h, w  = frames[0].shape[:2]
    M     = cv2.getRotationMatrix2D((w//2, h//2), angle, scale)
    return [cv2.warpAffine(f, M, (w, h)) for f in frames]

def standardize_frames(frames, seq_len=30):
    n = len(frames)
    if n >= seq_len:
        idxs = np.linspace(0, n-1, seq_len, dtype=int)
        return [frames[i] for i in idxs]
    return frames + [frames[-1]] * (seq_len - n)

def extract_keypoints(frame):
    rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

    hand_result = hand_landmarker.detect(mp_img)
    face_result = face_landmarker.detect(mp_img)

    # ── Hands ─────────────────────────────────────────────────────────────────
    rh = np.zeros((21, 3))
    lh = np.zeros((21, 3))
    if hand_result.hand_landmarks:
        for hand_lm, handedness in zip(
            hand_result.hand_landmarks,
            hand_result.handedness
        ):
            label = handedness[0].category_name   # "Left" or "Right"
            arr   = np.array([[p.x, p.y, p.z] for p in hand_lm])
            if label == "Right":
                rh = arr
            else:
                lh = arr

    # ── Face ──────────────────────────────────────────────────────────────────
    face_indices = [33, 263, 1, 61, 291, 199, 94, 0, 17, 57, 287]
    if face_result.face_landmarks:
        fl   = face_result.face_landmarks[0]
        face = np.array([[fl[i].x, fl[i].y, fl[i].z] for i in face_indices])
    else:
        face = np.zeros((11, 3))

    return np.concatenate([rh, lh, face], axis=0)   # shape (53, 3) — unchanged

def normalize_frame(kps_53x3):
    def norm_hand(h):
        h     = h - h[0:1]
        scale = np.linalg.norm(h.max(0) - h.min(0)) + 1e-8
        return h / scale
    def norm_face(f):
        center = (f[0] + f[1]) / 2
        dist   = np.linalg.norm(f[0] - f[1]) + 1e-8
        return (f - center) / dist
    rh   = norm_hand(kps_53x3[:21])
    lh   = norm_hand(kps_53x3[21:42])
    face = norm_face(kps_53x3[42:53])
    return np.concatenate([rh.flatten(), lh.flatten(), face.flatten()])

print("All functions defined ✓")

replace some videos that were not placed in the right place.

In [ ]:
import glob

src_dir = "/content/drive/MyDrive/arabic_dataset/KArSL-502/01/test/extracted/0005/"

files_to_move = glob.glob(src_dir + "*_16_34*_c.mp4") + glob.glob(src_dir + "*_17_11*_c.mp4")

print(f"Files that will be moved ({len(files_to_move)}):")
for f in sorted(files_to_move):
    print(os.path.basename(f))

In [ ]:
import shutil

dst_dir = "/content/drive/MyDrive/arabic_dataset/KArSL-502/01/test/extracted/0006/"
os.makedirs(dst_dir, exist_ok=True)

for f in sorted(files_to_move):
    dst_path = os.path.join(dst_dir, os.path.basename(f))
    shutil.move(f, dst_path)
    print(f"Moved: {os.path.basename(f)}")

print(f"\nDone! Moved {len(files_to_move)} files to 0006")

In [ ]:
src_dir = "/content/drive/MyDrive/arabic_dataset/KArSL-502/01/train/extracted/0005/"

files_to_move = (
    glob.glob(src_dir + "*_16_34*_c.mp4") +
    glob.glob(src_dir + "*_16_35*_c.mp4")
)

print(f"Files that will be moved ({len(files_to_move)}):")
for f in sorted(files_to_move):
    print(os.path.basename(f))

Data Augmentation for selected classes

In [ ]:
BASE_DIR = "/content/drive/MyDrive/arabic_dataset/KArSL-502"
OUT_DIR  = "/content/drive/MyDrive/arabic_dataset/NPY"

SIGNERS  = ["01", "02", "03"]
CLASSES  = (
    list(range(1, 11)) +
    list(range(32, 71)) +
    list(range(71, 81))
)

total_saved   = 0
total_skipped = 0

for signer in SIGNERS:
    for cls in CLASSES:
        cls_str = f"{cls:04d}"

        for split in ["train", "test"]:
            vid_dir = f"{BASE_DIR}/{signer}/{split}/extracted/{cls_str}/"
            out_dir = f"{OUT_DIR}/{split}/{cls_str}/{signer}/"
            os.makedirs(out_dir, exist_ok=True)

            videos = glob.glob(vid_dir + "*.mp4")
            if not videos:
                print(f"  WARNING: no videos at {vid_dir}")
                total_skipped += 1
                continue

            for vid_path in videos:
                vid_name = os.path.splitext(os.path.basename(vid_path))[0]
                frames   = read_video(vid_path)

                all_versions = [frames]
                if split == "train":
                    for i in range(N_AUG):
                        all_versions.append(augment_video(frames, seed=i))

                for v_idx, version in enumerate(all_versions):
                    out_path = f"{out_dir}/{vid_name}_aug{v_idx:02d}.npy"
                    if os.path.exists(out_path):
                        continue

                    std  = standardize_frames(version, SEQ_LEN)
                    kps  = [extract_keypoints(f) for f in std]   # ← no holistic argument
                    seq  = np.stack(kps)
                    norm = np.stack([normalize_frame(f) for f in seq])

                    np.save(out_path, norm)
                    total_saved += 1

        print(f"✓ Signer {signer} | Class {cls_str} | {split} | {len(videos)} videos")

print(f"\nDone! Saved: {total_saved} | Skipped: {total_skipped}")

Defining the load_split

In [ ]:
import numpy as np
import os
import glob
import gc  # >>> NEW: Garbage collection to manually free up RAM
from sklearn.model_selection import train_test_split

NPY_DIR = "/content/drive/MyDrive/arabic_dataset/NPY"
# >>> NEW: Define where you want to save the final processed arrays
SAVE_DIR = "/content/drive/MyDrive/arabic_dataset/Processed_Experiments"

# Map class folder names to integer labels 0-58
CLASSES = (
    list(range(1, 11)) +
    list(range(32, 71)) +
    list(range(71, 81))
)
CLASS_TO_LABEL = {f"{cls:04d}": idx for idx, cls in enumerate(CLASSES)}
print(f"Total classes: {len(CLASS_TO_LABEL)}")  # should be 59

def load_split(split, signers):
    """Load all .npy files for given split and signers."""
    X, y = [], []
    for cls_folder, label in CLASS_TO_LABEL.items():
        for signer in signers:
            pattern = f"{NPY_DIR}/{split}/{cls_folder}/{signer}/*.npy"
            files   = glob.glob(pattern)
            for f in files:
                # >>> NEW: Cast to float32 immediately to cut RAM usage in half
                seq = np.load(f).astype(np.float32)
                if seq.shape == (30, 159):
                    X.append(seq)
                    y.append(label)
                else:
                    print(f"  Bad shape {seq.shape}: {f}")
    return np.array(X), np.array(y)



Splitting the data to 3 experiments

In [ ]:
SIGNER_EXPERIMENTS = [
    {"train": ["01", "02"], "test": "03", "name": "Exp1_Test_03"},
    {"train": ["01", "03"], "test": "02", "name": "Exp2_Test_02"},
    {"train": ["02", "03"], "test": "01", "name": "Exp3_Test_01"},
]

for exp in SIGNER_EXPERIMENTS:
    print(f"\n--- Processing {exp['name']} ---")

    # Load train (augmented) from 2 signers
    X_train, y_train = load_split("train", exp["train"])

    # Load test (NOT augmented) from 1 unseen signer
    X_test,  y_test  = load_split("test",  [exp["test"]])

    # Split 10% of train as validation
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train, y_train,
        test_size=0.1,
        random_state=42,
        stratify=y_train   # keeps class balance in both halves
    )

    print(f"  Train:      {X_tr.shape}  labels: {y_tr.shape}")
    print(f"  Validation: {X_val.shape} labels: {y_val.shape}")
    print(f"  Test:       {X_test.shape} labels: {y_test.shape}")

    # >>> NEW: Save directly to Google Drive
    exp_folder = os.path.join(SAVE_DIR, exp["name"])
    os.makedirs(exp_folder, exist_ok=True)

    print(f"  Saving arrays to {exp_folder}...")
    np.save(os.path.join(exp_folder, "X_train.npy"), X_tr)
    np.save(os.path.join(exp_folder, "y_train.npy"), y_tr)
    np.save(os.path.join(exp_folder, "X_val.npy"), X_val)
    np.save(os.path.join(exp_folder, "y_val.npy"), y_val)
    np.save(os.path.join(exp_folder, "X_test.npy"), X_test)
    np.save(os.path.join(exp_folder, "y_test.npy"), y_test)

    # >>> NEW: Delete variables from RAM and force garbage collection
    print("  Clearing RAM for the next experiment...\n")
    del X_train, y_train, X_test, y_test, X_tr, X_val, y_tr, y_val
    gc.collect()

print("All experiments processed and saved to Drive successfully!")

Building the model

In [2]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

tf.random.set_seed(42)
np.random.seed(42)
print(f"TensorFlow version: {tf.__version__}")
def positional_encoding(seq_len, dim):
    pe  = np.zeros((seq_len, dim))
    pos = np.arange(seq_len)[:, np.newaxis]
    div = np.exp(np.arange(0, dim, 2) * -(np.log(10000.0) / dim))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div[:pe[:, 1::2].shape[1]])  # ← fix for odd dim
    return tf.cast(pe, dtype=tf.float32)

class PositionalEmbedding(keras.layers.Layer):
    def __init__(self, seq_len, dim, **kwargs):
        super().__init__(**kwargs)
        self.embed = keras.layers.Dense(dim)
        self.pe    = positional_encoding(seq_len, dim)

    def call(self, x):
        return self.embed(x) + self.pe

print("PositionalEmbedding defined ✓")
print(tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.20.0
PositionalEmbedding defined ✓


In [ ]:
def transformer_block(x, num_heads=4, key_dim=32, ff_dim=128, dropout=0.25):
    # Layer norm → multi-head attention → residual
    z = keras.layers.LayerNormalization(epsilon=1e-6)(x)
    a = keras.layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=key_dim)(z, z)
    z = keras.layers.Add()([x, a])

    # Feed forward → residual
    f = keras.layers.Dense(ff_dim, activation='relu')(z)
    f = keras.layers.Dense(x.shape[-1])(f)
    z = keras.layers.Add()([z, f])

    # Pool → dense → dropout
    z = keras.layers.GlobalAveragePooling1D()(z)
    z = keras.layers.Dense(64, activation='relu')(z)
    z = keras.layers.Dropout(dropout)(z)
    return z

print("transformer_block defined ✓")

In [ ]:
def gru_branch(x, dropout=0.5):
    g = keras.layers.GRU(64,  return_sequences=True)(x)
    g = keras.layers.GRU(128, return_sequences=True)(g)
    g = keras.layers.GRU(64,  return_sequences=False)(g)
    g = keras.layers.Dense(64, activation='relu')(g)
    g = keras.layers.Dropout(dropout)(g)
    return g

print("gru_branch defined ✓")

In [ ]:
def build_arsl_tgru(seq_len=30, feat_dim=159, n_classes=59):
    inputs = keras.Input(shape=(seq_len, feat_dim))

    # ── Transformer branch (4 sequential blocks) ──
    x = PositionalEmbedding(seq_len, feat_dim)(inputs)
    # After block 1, output is (batch, 64) — need to reshape for next block
    t = transformer_block(x)                    # (batch, 64)
    t = keras.layers.Reshape((1, 64))(t)        # (batch, 1, 64)
    t = transformer_block(t)                    # (batch, 64)
    t = keras.layers.Reshape((1, 64))(t)
    t = transformer_block(t)                    # (batch, 64)
    t = keras.layers.Reshape((1, 64))(t)
    t = transformer_block(t)                    # (batch, 64) — final

    # ── Two parallel GRU branches ──
    g1 = gru_branch(inputs)                     # (batch, 64)
    g2 = gru_branch(inputs)                     # (batch, 64)

    # ── Concatenate all three ──
    merged = keras.layers.Concatenate()([t, g1, g2])  # (batch, 192)

    # ── Classifier ──
    out = keras.layers.Dense(128, activation='relu')(merged)
    out = keras.layers.Dropout(0.5)(out)
    out = keras.layers.Dense(n_classes, activation='softmax')(out)

    model = keras.Model(inputs, out, name="ArSL_TGRU")
    return model

model = build_arsl_tgru()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled ✓")

Training Experiment 1

In [ ]:
# Load experiment 1
exp_folder = "/content/drive/MyDrive/arabic_dataset/Processed_Experiments/Exp1_Test_03"
X_train = np.load(f"{exp_folder}/X_train.npy")
y_train = np.load(f"{exp_folder}/y_train.npy")
X_val   = np.load(f"{exp_folder}/X_val.npy")
y_val   = np.load(f"{exp_folder}/y_val.npy")

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp1.h5",
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete ✓")

Epoch 1/50
3071/3071 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.2638 - loss: 2.6995
Epoch 1: val_loss improved from None to 0.39589, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp1.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 1: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp1.h5
3071/3071 ━━━━━━━━━━━━━━━━━━━━ 122s 34ms/step - accuracy: 0.4651 - loss: 1.7574 - val_accuracy: 0.8775 - val_loss: 0.3959 - learning_rate: 0.0010
Epoch 2/50
3071/3071 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.7658 - loss: 0.6721
Epoch 2: val_loss improved from 0.39589 to 0.18920, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp1.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 2: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp1.h5
3071/3071 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.8011 - loss: 0.5846 - val_accuracy: 0.9355 - val_loss: 0.1892 - learning_rate: 0.0010

test outputs

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

X_test = np.load(f"{exp_folder}/X_test.npy")
y_test = np.load(f"{exp_folder}/y_test.npy")

y_pred_prob = model.predict(X_test)
y_pred      = y_pred_prob.argmax(axis=1)

print("=== Exp1 — Unseen Signer 03 ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step
=== Exp1 — Unseen Signer 03 ===
Accuracy:  0.4767
Precision: 0.4413
Recall:    0.4767
F1-Score:  0.4212

loading exp 2

In [ ]:
exp_folder = "/content/drive/MyDrive/arabic_dataset/Processed_Experiments/Exp2_Test_02"
X_train = np.load(f"{exp_folder}/X_train.npy")
y_train = np.load(f"{exp_folder}/y_train.npy")
X_val   = np.load(f"{exp_folder}/X_val.npy")
y_val   = np.load(f"{exp_folder}/y_val.npy")
print("Loaded Exp2 ✓")

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("num classes:", len(np.unique(y_train)) if y_train.ndim == 1 else y_train.shape[1])

In [ ]:
#@title rebuilding model to start with random weights
model = build_arsl_tgru()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled ✓")

Training exp 2

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5",
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)
print("\nTraining complete ✓")

Epoch 1/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.2619 - loss: 2.6678
Epoch 1: val_loss improved from None to 0.56606, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 1: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 119s 34ms/step - accuracy: 0.4519 - loss: 1.7974 - val_accuracy: 0.8305 - val_loss: 0.5661 - learning_rate: 0.0010
Epoch 2/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7282 - loss: 0.8232
Epoch 2: val_loss improved from 0.56606 to 0.35924, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 2: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.7644 - loss: 0.7335 - val_accuracy: 0.8938 - val_loss: 0.3592 - learning_rate: 0.0010
Epoch 3/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8283 - loss: 0.5654
Epoch 3: val_loss improved from 0.35924 to 0.32360, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 3: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.8392 - loss: 0.5390 - val_accuracy: 0.9068 - val_loss: 0.3236 - learning_rate: 0.0010
Epoch 4/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8687 - loss: 0.4569
Epoch 4: val_loss did not improve from 0.32360
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.8722 - loss: 0.4497 - val_accuracy: 0.8947 - val_loss: 0.3488 - learning_rate: 0.0010
Epoch 5/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8820 - loss: 0.4197
Epoch 5: val_loss improved from 0.32360 to 0.23207, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 5: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.8862 - loss: 0.4075 - val_accuracy: 0.9356 - val_loss: 0.2321 - learning_rate: 0.0010
Epoch 6/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8900 - loss: 0.3934
Epoch 6: val_loss improved from 0.23207 to 0.20391, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 6: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.8952 - loss: 0.3754 - val_accuracy: 0.9434 - val_loss: 0.2039 - learning_rate: 0.0010
Epoch 7/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8987 - loss: 0.3660
Epoch 7: val_loss did not improve from 0.20391
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.8994 - loss: 0.3650 - val_accuracy: 0.9286 - val_loss: 0.2598 - learning_rate: 0.0010
Epoch 8/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9075 - loss: 0.3443
Epoch 8: val_loss did not improve from 0.20391
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9102 - loss: 0.3282 - val_accuracy: 0.9387 - val_loss: 0.2234 - learning_rate: 0.0010
Epoch 9/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9081 - loss: 0.3367
Epoch 9: val_loss improved from 0.20391 to 0.18696, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 9: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9132 - loss: 0.3202 - val_accuracy: 0.9498 - val_loss: 0.1870 - learning_rate: 0.0010
Epoch 10/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9121 - loss: 0.3213
Epoch 10: val_loss did not improve from 0.18696
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9151 - loss: 0.3154 - val_accuracy: 0.9316 - val_loss: 0.2517 - learning_rate: 0.0010
Epoch 11/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9198 - loss: 0.2952
Epoch 11: val_loss did not improve from 0.18696
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9197 - loss: 0.2989 - val_accuracy: 0.9496 - val_loss: 0.1933 - learning_rate: 0.0010
Epoch 12/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9227 - loss: 0.2829
Epoch 12: val_loss did not improve from 0.18696
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9217 - loss: 0.2891 - val_accuracy: 0.9359 - val_loss: 0.2393 - learning_rate: 0.0010
Epoch 13/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9152 - loss: 0.3111
Epoch 13: val_loss did not improve from 0.18696
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9219 - loss: 0.2889 - val_accuracy: 0.9409 - val_loss: 0.2082 - learning_rate: 0.0010
Epoch 14/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9252 - loss: 0.2816
Epoch 14: val_loss improved from 0.18696 to 0.18381, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 14: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.9254 - loss: 0.2796 - val_accuracy: 0.9511 - val_loss: 0.1838 - learning_rate: 0.0010
Epoch 15/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9232 - loss: 0.2888
Epoch 15: val_loss improved from 0.18381 to 0.17247, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 15: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9268 - loss: 0.2789 - val_accuracy: 0.9526 - val_loss: 0.1725 - learning_rate: 0.0010
Epoch 16/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9265 - loss: 0.2770
Epoch 16: val_loss did not improve from 0.17247
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9269 - loss: 0.2763 - val_accuracy: 0.9347 - val_loss: 0.2507 - learning_rate: 0.0010
Epoch 17/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9293 - loss: 0.2638
Epoch 17: val_loss did not improve from 0.17247
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 34ms/step - accuracy: 0.9319 - loss: 0.2550 - val_accuracy: 0.9482 - val_loss: 0.2004 - learning_rate: 0.0010
Epoch 18/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9259 - loss: 0.2822
Epoch 18: val_loss did not improve from 0.17247
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9269 - loss: 0.2778 - val_accuracy: 0.9489 - val_loss: 0.1835 - learning_rate: 0.0010
Epoch 19/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9288 - loss: 0.2754
Epoch 19: val_loss did not improve from 0.17247
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9312 - loss: 0.2634 - val_accuracy: 0.9453 - val_loss: 0.2172 - learning_rate: 0.0010
Epoch 20/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9311 - loss: 0.2622
Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 20: val_loss did not improve from 0.17247
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9334 - loss: 0.2538 - val_accuracy: 0.9518 - val_loss: 0.1869 - learning_rate: 0.0010
Epoch 21/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9423 - loss: 0.2130
Epoch 21: val_loss improved from 0.17247 to 0.15162, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 21: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.9464 - loss: 0.1972 - val_accuracy: 0.9591 - val_loss: 0.1516 - learning_rate: 5.0000e-04
Epoch 22/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9516 - loss: 0.1860
Epoch 22: val_loss did not improve from 0.15162
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9519 - loss: 0.1851 - val_accuracy: 0.9588 - val_loss: 0.1573 - learning_rate: 5.0000e-04
Epoch 23/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9503 - loss: 0.1845
Epoch 23: val_loss did not improve from 0.15162
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9505 - loss: 0.1843 - val_accuracy: 0.9573 - val_loss: 0.1559 - learning_rate: 5.0000e-04
Epoch 24/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9514 - loss: 0.1773
Epoch 24: val_loss did not improve from 0.15162
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9531 - loss: 0.1741 - val_accuracy: 0.9608 - val_loss: 0.1546 - learning_rate: 5.0000e-04
Epoch 25/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9508 - loss: 0.1840
Epoch 25: val_loss improved from 0.15162 to 0.14973, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 25: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.9519 - loss: 0.1800 - val_accuracy: 0.9599 - val_loss: 0.1497 - learning_rate: 5.0000e-04
Epoch 26/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9544 - loss: 0.1681
Epoch 26: val_loss did not improve from 0.14973
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9549 - loss: 0.1687 - val_accuracy: 0.9578 - val_loss: 0.1591 - learning_rate: 5.0000e-04
Epoch 27/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9526 - loss: 0.1785
Epoch 27: val_loss did not improve from 0.14973
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9530 - loss: 0.1784 - val_accuracy: 0.9546 - val_loss: 0.1655 - learning_rate: 5.0000e-04
Epoch 28/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9529 - loss: 0.1725
Epoch 28: val_loss did not improve from 0.14973
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9542 - loss: 0.1706 - val_accuracy: 0.9608 - val_loss: 0.1605 - learning_rate: 5.0000e-04
Epoch 29/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9530 - loss: 0.1769
Epoch 29: val_loss improved from 0.14973 to 0.14970, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 29: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9548 - loss: 0.1718 - val_accuracy: 0.9617 - val_loss: 0.1497 - learning_rate: 5.0000e-04
Epoch 30/50
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9537 - loss: 0.1691
Epoch 30: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 30: val_loss did not improve from 0.14970
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9540 - loss: 0.1716 - val_accuracy: 0.9558 - val_loss: 0.1706 - learning_rate: 5.0000e-04
Epoch 31/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9562 - loss: 0.1588
Epoch 31: val_loss did not improve from 0.14970
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9585 - loss: 0.1530 - val_accuracy: 0.9615 - val_loss: 0.1511 - learning_rate: 2.5000e-04
Epoch 32/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9591 - loss: 0.1529
Epoch 32: val_loss improved from 0.14970 to 0.14592, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 32: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp2.h5
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9596 - loss: 0.1499 - val_accuracy: 0.9617 - val_loss: 0.1459 - learning_rate: 2.5000e-04
Epoch 33/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9602 - loss: 0.1474
Epoch 33: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9611 - loss: 0.1440 - val_accuracy: 0.9615 - val_loss: 0.1489 - learning_rate: 2.5000e-04
Epoch 34/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9592 - loss: 0.1529
Epoch 34: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9608 - loss: 0.1471 - val_accuracy: 0.9613 - val_loss: 0.1502 - learning_rate: 2.5000e-04
Epoch 35/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9601 - loss: 0.1477
Epoch 35: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9612 - loss: 0.1437 - val_accuracy: 0.9617 - val_loss: 0.1581 - learning_rate: 2.5000e-04
Epoch 36/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9604 - loss: 0.1464
Epoch 36: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9612 - loss: 0.1447 - val_accuracy: 0.9613 - val_loss: 0.1613 - learning_rate: 2.5000e-04
Epoch 37/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9605 - loss: 0.1448
Epoch 37: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 37: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9613 - loss: 0.1419 - val_accuracy: 0.9619 - val_loss: 0.1542 - learning_rate: 2.5000e-04
Epoch 38/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9614 - loss: 0.1422
Epoch 38: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9628 - loss: 0.1376 - val_accuracy: 0.9630 - val_loss: 0.1469 - learning_rate: 1.2500e-04
Epoch 39/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9629 - loss: 0.1372
Epoch 39: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9638 - loss: 0.1344 - val_accuracy: 0.9628 - val_loss: 0.1475 - learning_rate: 1.2500e-04
Epoch 40/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9625 - loss: 0.1372
Epoch 40: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9629 - loss: 0.1352 - val_accuracy: 0.9617 - val_loss: 0.1554 - learning_rate: 1.2500e-04
Epoch 41/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.9623 - loss: 0.1371
Epoch 41: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9632 - loss: 0.1355 - val_accuracy: 0.9626 - val_loss: 0.1496 - learning_rate: 1.2500e-04
Epoch 42/50
3081/3082 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9619 - loss: 0.1384
Epoch 42: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 42: val_loss did not improve from 0.14592
3082/3082 ━━━━━━━━━━━━━━━━━━━━ 101s 33ms/step - accuracy: 0.9633 - loss: 0.1358 - val_accuracy: 0.9622 - val_loss: 0.1520 - learning_rate: 1.2500e-04
Epoch 42: early stopping
Restoring model weights from the end of the best epoch: 32.

Training complete ✓

Evaluating

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_test = np.load(f"{exp_folder}/X_test.npy")
y_test = np.load(f"{exp_folder}/y_test.npy")

y_pred_prob = model.predict(X_test)
y_pred      = y_pred_prob.argmax(axis=1)

print("=== Exp2 — Unseen Signer ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step
=== Exp2 — Unseen Signer ===
Accuracy:  0.6292
Precision: 0.6440
Recall:    0.6292
F1-Score:  0.6033

In [ ]:
exp_folder = "/content/drive/MyDrive/arabic_dataset/Processed_Experiments/Exp3_Test_01"
X_train = np.load(f"{exp_folder}/X_train.npy")
y_train = np.load(f"{exp_folder}/y_train.npy")
X_val   = np.load(f"{exp_folder}/X_val.npy")
y_val   = np.load(f"{exp_folder}/y_val.npy")
print("Loaded Exp3 ✓")

In [ ]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("num classes:", len(np.unique(y_train)) if y_train.ndim == 1 else y_train.shape[1])

In [ ]:
model = build_arsl_tgru()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("Model compiled ✓")

Training exp 3

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5",
        save_best_only=True,
        monitor='val_loss',
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=callbacks,
    verbose=1
)
print("\nTraining complete ✓")

Epoch 1/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.2423 - loss: 2.7811
Epoch 1: val_loss improved from None to 0.67732, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 1: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 118s 34ms/step - accuracy: 0.4221 - loss: 1.9151 - val_accuracy: 0.7676 - val_loss: 0.6773 - learning_rate: 0.0010
Epoch 2/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.6997 - loss: 0.8988
Epoch 2: val_loss improved from 0.67732 to 0.39930, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 2: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.7405 - loss: 0.7976 - val_accuracy: 0.8756 - val_loss: 0.3993 - learning_rate: 0.0010
Epoch 3/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8210 - loss: 0.5924
Epoch 3: val_loss did not improve from 0.39930
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.8291 - loss: 0.5658 - val_accuracy: 0.8728 - val_loss: 0.4152 - learning_rate: 0.0010
Epoch 4/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8541 - loss: 0.4950
Epoch 4: val_loss improved from 0.39930 to 0.32639, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 4: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.8611 - loss: 0.4763 - val_accuracy: 0.8931 - val_loss: 0.3264 - learning_rate: 0.0010
Epoch 5/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8770 - loss: 0.4296
Epoch 5: val_loss did not improve from 0.32639
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.8823 - loss: 0.4133 - val_accuracy: 0.9199 - val_loss: 0.3755 - learning_rate: 0.0010
Epoch 6/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8928 - loss: 0.3903
Epoch 6: val_loss improved from 0.32639 to 0.22621, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 6: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.8943 - loss: 0.3801 - val_accuracy: 0.9387 - val_loss: 0.2262 - learning_rate: 0.0010
Epoch 7/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9034 - loss: 0.3487
Epoch 7: val_loss improved from 0.22621 to 0.19359, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 7: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9021 - loss: 0.3545 - val_accuracy: 0.9444 - val_loss: 0.1936 - learning_rate: 0.0010
Epoch 8/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9086 - loss: 0.3366
Epoch 8: val_loss did not improve from 0.19359
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9048 - loss: 0.3473 - val_accuracy: 0.9445 - val_loss: 0.2036 - learning_rate: 0.0010
Epoch 9/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9177 - loss: 0.2996
Epoch 9: val_loss did not improve from 0.19359
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9160 - loss: 0.3085 - val_accuracy: 0.9454 - val_loss: 0.1962 - learning_rate: 0.0010
Epoch 10/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9153 - loss: 0.3175
Epoch 10: val_loss improved from 0.19359 to 0.19055, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 10: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9121 - loss: 0.3286 - val_accuracy: 0.9484 - val_loss: 0.1906 - learning_rate: 0.0010
Epoch 11/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9233 - loss: 0.2880
Epoch 11: val_loss did not improve from 0.19055
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9209 - loss: 0.2965 - val_accuracy: 0.9128 - val_loss: 0.3646 - learning_rate: 0.0010
Epoch 12/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9153 - loss: 0.3235
Epoch 12: val_loss improved from 0.19055 to 0.17185, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 12: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9189 - loss: 0.3072 - val_accuracy: 0.9524 - val_loss: 0.1719 - learning_rate: 0.0010
Epoch 13/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9214 - loss: 0.3018
Epoch 13: val_loss improved from 0.17185 to 0.16420, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 13: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9212 - loss: 0.3046 - val_accuracy: 0.9540 - val_loss: 0.1642 - learning_rate: 0.0010
Epoch 14/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9263 - loss: 0.2792
Epoch 14: val_loss did not improve from 0.16420
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9227 - loss: 0.2922 - val_accuracy: 0.9447 - val_loss: 0.2313 - learning_rate: 0.0010
Epoch 15/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9294 - loss: 0.2615
Epoch 15: val_loss did not improve from 0.16420
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9287 - loss: 0.2671 - val_accuracy: 0.9298 - val_loss: 0.3181 - learning_rate: 0.0010
Epoch 16/50
3082/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9271 - loss: 0.2786
Epoch 16: val_loss improved from 0.16420 to 0.14481, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 16: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9281 - loss: 0.2780 - val_accuracy: 0.9611 - val_loss: 0.1448 - learning_rate: 0.0010
Epoch 17/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9329 - loss: 0.2565
Epoch 17: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9305 - loss: 0.2645 - val_accuracy: 0.9575 - val_loss: 0.1502 - learning_rate: 0.0010
Epoch 18/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9316 - loss: 0.2527
Epoch 18: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9289 - loss: 0.2707 - val_accuracy: 0.9533 - val_loss: 0.1698 - learning_rate: 0.0010
Epoch 19/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9302 - loss: 0.2683
Epoch 19: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9324 - loss: 0.2569 - val_accuracy: 0.9524 - val_loss: 0.1802 - learning_rate: 0.0010
Epoch 20/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9285 - loss: 0.2835
Epoch 20: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9326 - loss: 0.2639 - val_accuracy: 0.9575 - val_loss: 0.1590 - learning_rate: 0.0010
Epoch 21/50
3082/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9342 - loss: 0.2522
Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 21: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9315 - loss: 0.2661 - val_accuracy: 0.9542 - val_loss: 0.1773 - learning_rate: 0.0010
Epoch 22/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9466 - loss: 0.2015
Epoch 22: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9487 - loss: 0.1941 - val_accuracy: 0.9584 - val_loss: 0.1770 - learning_rate: 5.0000e-04
Epoch 23/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9530 - loss: 0.1784
Epoch 23: val_loss did not improve from 0.14481
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9524 - loss: 0.1811 - val_accuracy: 0.9582 - val_loss: 0.1582 - learning_rate: 5.0000e-04
Epoch 24/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9522 - loss: 0.1850
Epoch 24: val_loss improved from 0.14481 to 0.14193, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 24: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9530 - loss: 0.1783 - val_accuracy: 0.9630 - val_loss: 0.1419 - learning_rate: 5.0000e-04
Epoch 25/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9562 - loss: 0.1642
Epoch 25: val_loss did not improve from 0.14193
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9544 - loss: 0.1729 - val_accuracy: 0.9631 - val_loss: 0.1444 - learning_rate: 5.0000e-04
Epoch 26/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9557 - loss: 0.1677
Epoch 26: val_loss improved from 0.14193 to 0.14070, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 26: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9546 - loss: 0.1704 - val_accuracy: 0.9628 - val_loss: 0.1407 - learning_rate: 5.0000e-04
Epoch 27/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9552 - loss: 0.1716
Epoch 27: val_loss did not improve from 0.14070
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9549 - loss: 0.1719 - val_accuracy: 0.9621 - val_loss: 0.1479 - learning_rate: 5.0000e-04
Epoch 28/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9561 - loss: 0.1676
Epoch 28: val_loss improved from 0.14070 to 0.13720, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 28: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9538 - loss: 0.1799 - val_accuracy: 0.9637 - val_loss: 0.1372 - learning_rate: 5.0000e-04
Epoch 29/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9573 - loss: 0.1591
Epoch 29: val_loss did not improve from 0.13720
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9558 - loss: 0.1668 - val_accuracy: 0.9628 - val_loss: 0.1422 - learning_rate: 5.0000e-04
Epoch 30/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9575 - loss: 0.1606
Epoch 30: val_loss did not improve from 0.13720
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9573 - loss: 0.1624 - val_accuracy: 0.9628 - val_loss: 0.1417 - learning_rate: 5.0000e-04
Epoch 31/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9556 - loss: 0.1725
Epoch 31: val_loss did not improve from 0.13720
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9554 - loss: 0.1726 - val_accuracy: 0.9551 - val_loss: 0.1825 - learning_rate: 5.0000e-04
Epoch 32/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9535 - loss: 0.1790
Epoch 32: val_loss improved from 0.13720 to 0.13399, saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
WARNING:absl:You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`.

Epoch 32: finished saving model to /content/drive/MyDrive/arabic_dataset/best_arsl_tgru_exp3.h5
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 104s 34ms/step - accuracy: 0.9550 - loss: 0.1723 - val_accuracy: 0.9633 - val_loss: 0.1340 - learning_rate: 5.0000e-04
Epoch 33/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9580 - loss: 0.1595
Epoch 33: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9567 - loss: 0.1680 - val_accuracy: 0.9582 - val_loss: 0.1771 - learning_rate: 5.0000e-04
Epoch 34/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9571 - loss: 0.1683
Epoch 34: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9570 - loss: 0.1660 - val_accuracy: 0.9635 - val_loss: 0.1402 - learning_rate: 5.0000e-04
Epoch 35/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9573 - loss: 0.1648
Epoch 35: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9560 - loss: 0.1708 - val_accuracy: 0.9646 - val_loss: 0.1444 - learning_rate: 5.0000e-04
Epoch 36/50
3082/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9588 - loss: 0.1610
Epoch 36: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9579 - loss: 0.1636 - val_accuracy: 0.9639 - val_loss: 0.1430 - learning_rate: 5.0000e-04
Epoch 37/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9579 - loss: 0.1704
Epoch 37: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 37: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9574 - loss: 0.1646 - val_accuracy: 0.9646 - val_loss: 0.1374 - learning_rate: 5.0000e-04
Epoch 38/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9620 - loss: 0.1460
Epoch 38: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9613 - loss: 0.1466 - val_accuracy: 0.9659 - val_loss: 0.1348 - learning_rate: 2.5000e-04
Epoch 39/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9618 - loss: 0.1439
Epoch 39: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9614 - loss: 0.1451 - val_accuracy: 0.9653 - val_loss: 0.1381 - learning_rate: 2.5000e-04
Epoch 40/50
3082/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9625 - loss: 0.1388
Epoch 40: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9618 - loss: 0.1413 - val_accuracy: 0.9661 - val_loss: 0.1371 - learning_rate: 2.5000e-04
Epoch 41/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9635 - loss: 0.1362
Epoch 41: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 103s 33ms/step - accuracy: 0.9626 - loss: 0.1371 - val_accuracy: 0.9648 - val_loss: 0.1414 - learning_rate: 2.5000e-04
Epoch 42/50
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.9639 - loss: 0.1378
Epoch 42: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 42: val_loss did not improve from 0.13399
3083/3083 ━━━━━━━━━━━━━━━━━━━━ 102s 33ms/step - accuracy: 0.9634 - loss: 0.1393 - val_accuracy: 0.9653 - val_loss: 0.1427 - learning_rate: 2.5000e-04
Epoch 42: early stopping
Restoring model weights from the end of the best epoch: 32.

Training complete ✓


Evaluating

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_test = np.load(f"{exp_folder}/X_test.npy")
y_test = np.load(f"{exp_folder}/y_test.npy")

y_pred_prob = model.predict(X_test)
y_pred      = y_pred_prob.argmax(axis=1)

print("=== Exp3 — Unseen Signer 01 ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 61ms/step
=== Exp3 — Unseen Signer 01 ===
Accuracy:  0.5000
Precision: 0.5266
Recall:    0.5000
F1-Score:  0.4720